# Train the Mitama's voice

This retrains MitamaOS's on-device model from scratch on public-domain myth and folklore, the D&D 5e SRD, a little code, and the Mitama's own lines, then writes a `model.bin` for the Cardputer.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order (Runtime > Run all works too). The whole thing takes about an hour on a T4.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the runtime to T4 GPU, or this will be very slow'

## 1. Get the kit

In [ ]:
!git clone -q --depth 1 -b claude/cardputer-adv-os-firmware-okqopr https://github.com/Xorbret/Xorbret.git
%cd Xorbret/mitamaos/voice

## 2. Build the training text
Downloads the books and SRD data (a few MB) and writes `data/pretrain.txt` and `data/voice.txt`.

In [ ]:
!python build_corpus.py

## 3. Train
First it learns English and the lore (pretrain), then the Mitama's voice (finetune). Every 2000 steps it prints a few sample answers so you can watch the voice form. Fewer steps = faster but blunter.

In [ ]:
SIZE = "small"  #@param ["small", "big"]
# small: 1.7 MB, lives in flash, fast. big: 4.7 MB, read from the SD card, slower but smarter.
PRETRAIN_STEPS = 20000  #@param {type:"integer"}
FINETUNE_STEPS = 3000   #@param {type:"integer"}
!python train.py --size {SIZE} --pretrain-steps {PRETRAIN_STEPS} --finetune-steps {FINETUNE_STEPS}

## 4. Check and preview
Runs `model.bin` with the same int8 maths and sampling as the Cardputer.

In [ ]:
!python check_export.py out/mitama.pt out/model.bin
!python modelbin.py out/model.bin -n 3 --prompt "Q: who are you?\nA:" --prompt "Q: what is a kitsune?\nA:" --prompt "Q: *idle*\nA:" --prompt "Q: *seen AIRTAG*\nA:"

## 5. Download
Copy the file to the SD card as `/MitamaOS/model.bin` (keep your old one somewhere first, in case you prefer it).

In [ ]:
from google.colab import files
files.download('out/model.bin')

### Only want more voice?
To retune the voice without redoing the pretrain, edit `mitama_voice.txt`, rerun step 2, then:

`!python train.py --size {SIZE} --resume out/pretrain.pt --pretrain-steps 0 --finetune-steps 3000`